# 10 · Score mechanics: how the rubric turns findings into a score

From Code Compliance's scoring criteria (`_background/Graded Inspection Scoring Criteria`):

- score = exterior (max 60) + average interior unit score (max 40);
- interior units inspected = 10% of units or 3, whichever is greater;
- a life hazard (no AC, no heat, no hot water, exposed electrical) scores that unit zero;
- other violations deduct 1–4 points by severity, which the inspector judges.

This notebook works out what that design implies and checks it against the data. How much
does one bad unit move the score at different property sizes? How likely is the sample to
find it? How close do scores sit to the bottom-10 cut (86) and the schedule bands (80, 90)?

## Findings

Run 2026-10-05. Outputs are cleared before commit; the numbers below are from that run.

**At small properties a single unit swings the score more than the whole spread between
inspectors. Scores pile up at 100, so bottom-10 status at the margin rests on a deduction
or two.**

- **Rubric arithmetic.** Up to 30 units, inspectors sample 3 units, and one zeroed unit
  costs 13.3 points. That is more than the 10.9-point spread between the 10th and 90th
  percentile inspector. One zeroed unit costs 4.0 points at 100 units and 1.3 points at 300.
- **Small samples usually miss a single bad unit.** The chance the sample finds one hazard
  unit is 75% at 4 units, 15% at 20, and 10% at 30 or more. If 5% of units are hazardous, the
  chance of finding any is 19% at 30–50 units, 42% at 100 and 80% at 300.
- **By size in the data.**
  - The bottom-10 rate is highest at 21–50 units (15%), against 8% above 150 units. Score SD
    runs 11.6–14.4 across size bands.
  - Inspections with a flagged life-safety hazard are bottom-10 2–5 times as often (16–46%,
    against 7–15%). The score gap doesn't follow the rubric's size gradient, though: 4.2
    points at 20 units or fewer, where the rubric implies 13.3; 23 at 21–50; 0.8 at 51–150.
    With 13–43 flagged inspections per band, and flags that may sit on exterior or
    common-area items, the reason is unclear.
  - The same-inspector gap holds at every size: score r 0.25–0.58 with the same inspector,
    against 0.00–0.11 with a different one.
- **Scores pile up at the top.** 41.7% of inspections score exactly 100, 57.5% score 98 or
  more, and 85.8% fall in the 90+ (three-year) band.
  - 28.6% of bottom-10 inspections sit within 2 points of the cut (84–86), and 40.6% within 4.
  - 10% of all inspections sit within 2 points of a schedule band edge (80 or 90).
- **What this means.** At small properties, which units are sampled and which checklist item
  is used can outweigh the inspector. At large properties, how severely the inspector grades
  dominates.

In [ ]:
from __future__ import annotations

import math

import numpy as np
import pandas as pd

import holdout as h
import reliability as rl
import temporal as t

pd.set_option("display.float_format", "{:.3f}".format)
insp = t.load_inspections()
insp = insp[insp.inspector >= 0].copy()

## 1. Rubric arithmetic

In [ ]:
def sample_size(n: int) -> int:
    return max(3, math.ceil(0.1 * n))


def p_found(n: int, bad: int) -> float:
    # chance a random sample of units includes at least one of `bad` hazard units
    k = sample_size(n)
    if bad <= 0:
        return 0.0
    if n - bad < k:
        return 1.0
    return 1 - math.comb(n - bad, k) / math.comb(n, k)


rows = []
for n in [4, 8, 12, 20, 30, 50, 100, 200, 300, 500]:
    k = sample_size(n)
    five = max(1, round(0.05 * n))
    rows.append({"units": n, "units inspected": k, "points lost if one sampled unit is zeroed": 40 / k,
                 "P(sample finds 1 hazard unit)": p_found(n, 1),
                 "hazard units = 5% of units": five, "P(sample finds any of them)": p_found(n, five)})
print(pd.DataFrame(rows).round(2).to_string(index=False))

## 2. What the data show by property size

In [ ]:
obs = rl.tier_observations()
true_lh = obs[obs.tier == "life_safety"].groupby("sr_number").life_hazard.max()
insp["true_life_hazard"] = insp.sr_number.map(true_lh).fillna(0)
insp["size"] = pd.cut(insp.n_units, [0, 20, 50, 150, 300, 1e6], labels=["≤20", "21-50", "51-150", "151-300", "300+"])

pairs = rl.consecutive_pairs(insp)
pairs["size"] = pd.cut(pairs.n_units, [0, 20, 50, 150, 300, 1e6], labels=["≤20", "21-50", "51-150", "151-300", "300+"])
rows = []
for s, d in insp.groupby("size", observed=True):
    lh, no = d[d.true_life_hazard == 1], d[d.true_life_hazard == 0]
    p = pairs[pairs["size"] == s]
    rows.append({
        "size": s, "inspections": len(d), "median units": d.n_units.median(),
        "rubric points per zeroed unit": 40 / sample_size(int(d.n_units.median())),
        "bottom-10 rate": d.bottom10.mean(), "score SD": d.score.std(),
        "inspections with a flagged life-safety hazard": len(lh),
        "mean score gap, flagged vs not": no.score.mean() - lh.score.mean(),
        "P(bottom-10 | flagged)": lh.bottom10.mean(), "P(bottom-10 | not)": no.bottom10.mean(),
        "consecutive r, same inspector": p[p.same_inspector].score.corr(p[p.same_inspector].prev_score),
        "consecutive r, different": p[~p.same_inspector].score.corr(p[~p.same_inspector].prev_score),
    })
print(pd.DataFrame(rows).round(2).T.to_string())

## 3. How close scores sit to the cut-offs

In [ ]:
s = insp.score
print(f"bottom-10 cut: score <= 86. Share of inspections scoring 85-87: {s.between(85, 87).mean():.1%}; "
      f"84-88: {s.between(84, 88).mean():.1%}")
b10 = insp[insp.bottom10 == 1]
print(f"of bottom-10 inspections, share within 2 points of the cut (84-86): {b10.score.between(84, 86).mean():.1%}; "
      f"within 4 (82-86): {b10.score.between(82, 86).mean():.1%}")
print(f"share scoring exactly 100: {(s == 100).mean():.1%}; 98 or above: {(s >= 98).mean():.1%}")
print(f"schedule bands: under 80 {(s < 80).mean():.1%}, 80-90 {s.between(80, 90, inclusive='left').mean():.1%}, "
      f"90+ {(s >= 90).mean():.1%}; within 2 points of 80 or 90: {(s.between(78, 82) | s.between(88, 92)).mean():.1%}")
print("\nmost common scores:", s.round(1).value_counts().head(8).to_dict())

## 4. A single-unit swing against the inspector spread

The inspector effect from notebook 08 (SD 4.9 points, about 11 points from the 10th to
the 90th percentile inspector) set against what one zeroed unit costs at each size.

In [ ]:
eff = rl.two_way_effects(insp)["rater"]
big = insp.inspector.value_counts().loc[lambda v: v >= 30].index
spread = np.percentile(eff.reindex(big), 90) - np.percentile(eff.reindex(big), 10)
print(f"10th-90th percentile inspector spread: {spread:.1f} points")
for n in [8, 30, 100, 300]:
    print(f"  {n:>3} units: one zeroed unit costs {40 / sample_size(n):.1f} points "
          f"({40 / sample_size(n) / spread:.0%} of the inspector spread)")